In [ ]:
from dataclasses import dataclass
from os import getenv
from pathlib import Path

from aizynthfinder.aizynthfinder import AiZynthExpander, AiZynthFinder
from aizynthfinder.interfaces import AiZynthApp
from aizynthfinder.interfaces.gui.clustering import ClusteringGui
from aizynthfinder.chem.reaction import FixedRetroReaction
from aizynthfinder.chem.mol import Molecule
from aizynthfinder.utils.logging import setup_logger

from dotenv import load_dotenv

load_dotenv()


CONFIG_PATH = Path(getenv("AZF_CONFIG_PATH"))

In [2]:
@dataclass
class RetroConfig:
    n_routes: int = 5
    minutes: int = 1
    seconds: int = minutes * 60
    depth: int = 10
    stock: str | list[str] = "zinc"
    expansion_policy: str | list[str] = "uspto"
    filter_policy: str | list[str] = "uspto"

m_config = RetroConfig()

## AiZynthApp

In [ ]:
application = AiZynthApp(CONFIG_PATH)

In [ ]:
%matplotlib inline
ClusteringGui.from_app(application)

## AiZynthFinder

In [3]:
setup_logger("WARNING")
finder = AiZynthFinder(CONFIG_PATH)

In [ ]:
target_mol = Molecule(smiles="O=C(C(CC1)CCN1S(c(cc1)ccc1F)(=O)=O)NCCCN1CCOCC1")
target_mol.rd_mol

In [ ]:
finder.config.__dict__

In [ ]:
setup_logger("INFO")
finder.target_mol = target_mol
finder.stock.select(m_config.stock)
finder.expansion_policy.select(m_config.expansion_policy)
finder.filter_policy.select(m_config.filter_policy)

finder.config.search.time_limit = m_config.seconds

In [ ]:
finder.prepare_tree()
finder.tree_search(show_progress=True)

In [ ]:
finder.build_routes()

In [ ]:
finder.extract_statistics()

In [ ]:
finder.routes.images[2]

In [ ]:
finder.routes[0]

## AiZynthExpander

In [ ]:
setup_logger("WARNING")
expander = AiZynthExpander(CONFIG_PATH)

In [ ]:
setup_logger("INFO")
expander.expansion_policy.select(m_config.expansion_policy)

In [ ]:
o_config = RetroConfig()
target_mol = Molecule(smiles="O=C(C(CC1)CCN1S(c(cc1)ccc1F)(=O)=O)NCCCN1CCOCC1")
target_mol.rd_mol

In [ ]:
reactions = expander.do_expansion(
    target_mol.smiles,
    o_config.n_routes,
)

In [ ]:
for reaction in reactions:
    example: FixedRetroReaction = reaction[0]
    children = [mol for mol in example.reactants[0]]
    metadata = example.metadata
    print("-" * 70)
    display(*[mol.rd_mol for mol in children])
    # display(metadata)